In [96]:
import pandas as pd
import numpy as np

datos = pd.read_excel("[HackMTY2025]_ConsumptionPrediction_Dataset_v1.xlsx")

In [97]:
datos.head()

,Flight_ID,Origin,Date,Flight_Type,Service_Type,Passenger_Count,Product_ID,Product_Name,Standard_Specification_Qty,Quantity_Returned,Quantity_Consumed,Unit_Cost,Crew_Feedback
0,AM109,DOH,2025-09-26,medium-haul,Retail,272,BRD001,Bread Roll Pack,62,7,55,0.35,NaN
1,AM109,DOH,2025-09-26,medium-haul,Retail,272,CRK075,Butter Cookies 75g,74,14,60,0.75,NaN
2,AM109,DOH,2025-09-26,medium-haul,Retail,272,DRK023,Sparkling Water 330ml,125,30,95,0.45,NaN
3,AM109,DOH,2025-09-26,medium-haul,Retail,272,DRK024,Still Water 500ml,110,19,91,0.50,NaN
4,LX110,DOH,2025-09-26,medium-haul,Pick & Pack,272,BRD001,Bread Roll Pack,177,58,119,0.35,NaN


In [98]:
print(datos.isnull().sum())

Flight_ID                       0
Origin                          0
Date                            0
Flight_Type                     0
Service_Type                    0
Passenger_Count                 0
Product_ID                      0
Product_Name                    0
Standard_Specification_Qty      0
Quantity_Returned               0
Quantity_Consumed               0
Unit_Cost                       0
Crew_Feedback                 723
dtype: int64


In [99]:
datos.drop('Crew_Feedback', axis=1, inplace=True)
datos.drop('Flight_ID', axis=1, inplace=True)
datos.drop('Service_Type', axis=1, inplace=True)
datos.drop('Product_ID', axis=1, inplace=True)
datos.drop('Unit_Cost', axis=1, inplace=True)

In [100]:
print(datos.isnull().sum())

Origin                        0
Date                          0
Flight_Type                   0
Passenger_Count               0
Product_Name                  0
Standard_Specification_Qty    0
Quantity_Returned             0
Quantity_Consumed             0
dtype: int64


In [101]:
print(datos.columns)

Index(['Origin', 'Date', 'Flight_Type', 'Passenger_Count', 'Product_Name',
       'Standard_Specification_Qty', 'Quantity_Returned', 'Quantity_Consumed'],
      dtype='object')


In [102]:
# Checar datos para resolver error

print("Columnas disponibles:", datos.columns.to_list())

Columnas disponibles: ['Origin', 'Date', 'Flight_Type', 'Passenger_Count', 'Product_Name', 'Standard_Specification_Qty', 'Quantity_Returned', 'Quantity_Consumed']


In [103]:
# Convertir la columna 'Date' a formato de fecha
datos['Date'] = pd.to_datetime(datos['Date'])

# Extraer características relevantes
datos['Year'] = datos['Date'].dt.year
datos['Month'] = datos['Date'].dt.month
datos['DayOfWeek'] = datos['Date'].dt.dayofweek # Lunes=0, Domingo=6

datos_originales_para_ver = datos[['Date', 'Origin', 'Flight_Type', 'Passenger_Count']].copy()

datos = datos.drop('Date', axis=1) # Ya no necesitamos la columna original

In [104]:
# 'Product_Name' es uno de nuestros objetivos (target), lo manejaremos después.
# Primero codificamos las características de entrada (features).
datos = pd.get_dummies(datos, columns=['Origin', 'Flight_Type'])

In [105]:
# Convertimos el nombre del producto a un código numérico (esto será nuestra salida categórica)
datos['Product_Code'] = datos['Product_Name'].astype('category').cat.codes
product_mapping = dict(enumerate(datos['Product_Name'].astype('category').cat.categories)) # Guardamos el mapeo para después

# Separar las salidas
y_producto = datos['Product_Code']
y_cantidad = datos['Quantity_Consumed']

# Separar las entradas y eliminar las columnas que son 'targets' o ya no sirven
X = datos.drop(['Product_Name', 'Product_Code', 'Quantity_Consumed', 'Quantity_Returned', 'Standard_Specification_Qty'], axis=1)

In [106]:
from sklearn.model_selection import train_test_split

# --- MODIFICACIÓN 2: Añade 'datos_originales_para_ver' ---
X_train, X_test, \
y_producto_train, y_producto_test, \
y_cantidad_train, y_cantidad_test, \
orig_train, orig_test = train_test_split(
    X, y_producto, y_cantidad, datos_originales_para_ver, # <--- ¡Añadido aquí!
    test_size=0.2, random_state=42
)

trained_model_columns = X_train.columns.to_list()
# 'orig_test' ahora contiene las entradas originales para tu conjunto de prueba
# -----------------------------------------------------------

In [107]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout

# Definimos las entradas del modelo
input_layer = Input(shape=(X_train.shape[1],), name='input_features')

# Capas ocultas (el "cerebro" del modelo)
shared_layers = Dense(128, activation='relu')(input_layer)
shared_layers = Dropout(0.2)(shared_layers) # Dropout ayuda a prevenir sobreajuste
shared_layers = Dense(64, activation='relu')(shared_layers)

# ---- Cabeza de Salida 1: Predicción del Producto (Clasificación) ----
num_products = len(product_mapping) # Número de productos únicos
product_output = Dense(num_products, activation='softmax', name='product_output')(shared_layers)

# ---- Cabeza de Salida 2: Predicción de la Cantidad (Regresión) ----
quantity_output = Dense(1, activation='linear', name='quantity_output')(shared_layers)

# Unimos todo en un solo modelo
model = Model(inputs=input_layer, outputs=[product_output, quantity_output])

# Vemos un resumen de la arquitectura
model.summary()

Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_features      │ (None, 13)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_14 (Dense)    │ (None, 128)       │      1,792 │ input_features[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 128)       │          0 │ dense_14[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_15 (Dense)    │ (None, 64)        │      8,256 │ dropout_7[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ product_output      │ (None, 10)        │        650 │ dense_15[0][0]    │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ quantity_output     │ (None, 1)         │         65 │ dense_15[0][0]    │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 10,763 (42.04 KB)

 Trainable params: 10,763 (42.04 KB)

 Non-trainable params: 0 (0.00 B)

In [108]:
### Test de datos
# --- INSERTA ESTE BLOQUE ANTES DE model.compile() ---

# 1. Revisar si hay NaNs (Nulos)
print("--- Revisando valores nulos (NaN) ---")
print(f"NaNs en X_train: {X_train.isnull().sum().sum()}")
print(f"NaNs en y_producto_train: {y_producto_train.isnull().sum()}")
print(f"NaNs en y_cantidad_train: {y_cantidad_train.isnull().sum()}")

# 2. Limpiar los NaNs
# Rellenamos los valores faltantes con 0. 
# (Puedes elegir otra estrategia, pero '0' es la más simple para empezar)
print("\n--- Limpiando NaNs (rellenando con 0) ---")
X_train = X_train.fillna(0)
X_test = X_test.fillna(0) # ¡Importante! Haz lo mismo para los datos de test

y_cantidad_train = y_cantidad_train.fillna(0)
y_cantidad_test = y_cantidad_test.fillna(0) # ¡Importante!

# y_producto no debería tener NaNs gracias a .cat.codes, pero lo revisamos por si acaso.
y_producto_train = y_producto_train.fillna(0) 
y_producto_test = y_producto_test.fillna(0) 


# 3. Asegurar los tipos de datos (DTypes)
# Forzamos los tipos de datos que Keras espera.
print("--- Forzando tipos de datos (dtypes) ---")
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')

# 'sparse_categorical_crossentropy' necesita enteros
y_producto_train = y_producto_train.astype('int32')
y_producto_test = y_producto_test.astype('int32')

# 'mean_squared_error' (regresión) funciona mejor con flotantes
y_cantidad_train = y_cantidad_train.astype('float32')
y_cantidad_test = y_cantidad_test.astype('float32')

print("\n¡Limpieza de datos completa! Listo para entrenar.")
# --- FIN DEL BLOQUE ---

# ... (el resto de tu código)

--- Revisando valores nulos (NaN) ---
NaNs en X_train: 0
NaNs en y_producto_train: 0
NaNs en y_cantidad_train: 0

--- Limpiando NaNs (rellenando con 0) ---
--- Forzando tipos de datos (dtypes) ---

¡Limpieza de datos completa! Listo para entrenar.


In [109]:
# Compilamos el modelo con dos funciones de pérdida
model.compile(optimizer='adam',
              loss={
                  'product_output': 'sparse_categorical_crossentropy',
                  'quantity_output': 'mean_squared_error'
              },
              metrics={
                  'product_output': 'accuracy' # Nos interesa la precisión para el producto
              })

# Entrenamos el modelo
history = model.fit(
    X_train,
    {'product_output': y_producto_train, 'quantity_output': y_cantidad_train},
    epochs=50,  # Número de veces que el modelo verá todos los datos
    batch_size=32,
    validation_split=0.2 # Usamos parte de los datos de entrenamiento para validar
)

Epoch 1/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - loss: 10407.5664 - product_output_accuracy: 0.0949 - product_output_loss: 295.1389 - quantity_output_loss: 10174.8848 - val_loss: 3874.9275 - val_product_output_accuracy: 0.0866 - val_product_output_loss: 185.5402 - val_quantity_output_loss: 3692.5488
Epoch 2/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 6163.8096 - product_output_accuracy: 0.1028 - product_output_loss: 195.3191 - quantity_output_loss: 5942.5444 - val_loss: 1588.4232 - val_product_output_accuracy: 0.0866 - val_product_output_loss: 125.1064 - val_quantity_output_loss: 1462.1776
Epoch 3/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 4564.0225 - product_output_accuracy: 0.1285 - product_output_loss: 147.7307 - quantity_output_loss: 4407.0898 - val_loss: 1822.2240 - val_product_output_accuracy: 0.0866 - val_product_output_loss: 76.1544 - val_quantity_output_loss: 1744.1500
Epoch 4/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 3607.4695 - product_output_accur

In [110]:
# Evaluar el modelo (tu código original)
loss, product_loss, quantity_loss, product_accuracy = model.evaluate(
    X_test, [y_producto_test, y_cantidad_test]
)
print(f"Precisión del producto: {product_accuracy * 100:.2f}%")
print(f"Error en la cantidad (MSE): {quantity_loss}")

# Para hacer una predicción con datos nuevos
predicciones = model.predict(X_test)
predicted_products = np.argmax(predicciones[0], axis=1)
predicted_quantities = predicciones[1]

# --- VERSIÓN MEJORADA DEL BUCLE ---
print("\n--- Mostrando las primeras 5 predicciones detalladas ---")

# Usamos .iloc para iterar por posición (0, 1, 2, 3, 4)
# Todos los dataframes de prueba ('orig_test', 'y_producto_test', etc.)
# están alineados por el 'train_test_split'.
for i in range(5):
    
    # 1. Obtenemos las entradas ORIGINALES
    # .iloc[i] toma la i-ésima fila de nuestros datos de prueba originales
    inputs_originales = orig_test.iloc[i]
    
    # 2. Obtenemos los resultados REALES
    real_product_name = product_mapping[y_producto_test.iloc[i]]
    real_quantity = y_cantidad_test.iloc[i]
    
    # 3. Obtenemos los resultados PREDICHOS
    predicted_product_name = product_mapping[predicted_products[i]]
    predicted_quantity = predicted_quantities[i][0] # [i][0] para sacar el número

    # 4. Imprimimos todo de forma clara
    print("=====================================================")
    print(f"PREDICCIÓN #{i+1}")
    print("=====================================================")
    
    print("--- DATOS DE ENTRADA (Originales) ---")
    print(f"  Fecha:         {inputs_originales['Date'].strftime('%Y-%m-%d')}")
    print(f"  Origen:        {inputs_originales['Origin']}")
    print(f"  Tipo de Vuelo: {inputs_originales['Flight_Type']}")
    print(f"  Pasajeros:     {inputs_originales['Passenger_Count']}")
    print("\n--- RESULTADOS DEL MODELO ---")
    print(f"  PRODUCTO:")
    print(f"    -> Real:    '{real_product_name}'")
    print(f"    -> Predicho: '{predicted_product_name}'")
    print(f"\n  CANTIDAD:")
    print(f"    -> Real:    {real_quantity}")
    print(f"    -> Predicha: {predicted_quantity:.2f}") # Redondeamos a 2 decimales
    print("\n")

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 1077.4441 - product_output_accuracy: 0.1069 - product_output_loss: 7.4092 - quantity_output_loss: 1074.0925
Precisión del producto: 10.69%
Error en la cantidad (MSE): 1074.092529296875
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step

--- Mostrando las primeras 5 predicciones detalladas ---
PREDICCIÓN #1
--- DATOS DE ENTRADA (Originales) ---
  Fecha:         2025-10-06
  Origen:        LHR
  Tipo de Vuelo: short-haul
  Pasajeros:     161

--- RESULTADOS DEL MODELO ---
  PRODUCTO:
    -> Real:    'Still Water 500ml'
    -> Predicho: 'Bread Roll Pack'

  CANTIDAD:
    -> Real:    53.0
    -> Predicha: 67.83


PREDICCIÓN #2
--- DATOS DE ENTRADA (Originales) ---
  Fecha:         2025-10-04
  Origen:        DOH
  Tipo de Vuelo: medium-haul
  Pasajeros:     294

--- RESULTADOS DEL MODELO ---
  PRODUCTO:
    -> Real:    'Sparkling Water 330ml'
    -> Predicho: 'Sparkling Water 330ml'

  CANTIDAD:
    -> Real:    192.0
    -> Predicha: 115.86


PREDICCIÓN #3


In [111]:
# --- CELDA 2 (EJECÚTALA) ---

def preparar_datos_para_prediccion(raw_data_dict, model_columns):
    """
    Toma un diccionario de datos crudos (humanos) y los prepara
    para que el modelo pueda hacer una predicción.
    """
    # 1. Convertir el diccionario a un DataFrame
    df = pd.DataFrame([raw_data_dict])

    # 2. Procesar la Fecha (igual que en el entrenamiento)
    df['Date'] = pd.to_datetime(df['Date'])
    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month
    df['DayOfWeek'] = df['Date'].dt.dayofweek
    df = df.drop('Date', axis=1) # Eliminar la fecha original

    # 3. Aplicar One-Hot Encoding (get_dummies)
    df = pd.get_dummies(df, columns=['Origin', 'Flight_Type'])

    # 4. Alineación de Columnas (¡EL PASO MÁS IMPORTANTE!)
    df = df.reindex(columns=model_columns, fill_value=0)

    # 5. Asegurar el tipo de dato
    df = df.astype('float32')
    
    return df

In [113]:
# --- BLOQUE PARA PROBAR DATOS NUEVOS (FORMATO MEJORADO) ---

print("\n\n========================================")
print("     PRUEBA CON DATOS 'JUPITER'     ")
print("========================================")

# 1. Define tus datos de prueba
datos_de_prueba = {
    'Origin': 'JFK',
    'Date': '2025-10-27',
    'Flight_Type': 'long-haul',
    'Passenger_Count': 350
}

# 2. Prepara los datos
datos_listos = preparar_datos_para_prediccion(
    datos_de_prueba, 
    trained_model_columns 
)

# 3. Haz la predicción
prediccion = model.predict(datos_listos)

# 4. Decodifica los resultados
nombre_producto = product_mapping[np.argmax(prediccion[0], axis=1)[0]]
cantidad_predicha = prediccion[1][0][0]

# 5. Muestra el resultado (con el formato que te gustó)
print("--- DATOS DE ENTRADA (Nuevos) ---")
print(f"  Fecha:         {datos_de_prueba['Date']}")
print(f"  Origen:        {datos_de_prueba['Origin']}")
print(f"  Tipo de Vuelo: {datos_de_prueba['Flight_Type']}")
print(f"  Pasajeros:     {datos_de_prueba['Passenger_Count']}")

print("\n--- RESULTADOS DEL MODELO ---")
print(f"  PRODUCTO:")
# No hay "Real" porque son datos nuevos
print(f"    -> Predicho: '{nombre_producto}'") 
print(f"\n  CANTIDAD:")
# No hay "Real" porque son datos nuevos
print(f"    -> Predicha: {cantidad_predicha:.2f} unidades")
print("\n")



     PRUEBA CON DATOS 'JUPITER'     
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
--- DATOS DE ENTRADA (Nuevos) ---
  Fecha:         2025-10-27
  Origen:        JFK
  Tipo de Vuelo: long-haul
  Pasajeros:     350

--- RESULTADOS DEL MODELO ---
  PRODUCTO:
    -> Predicho: 'Sparkling Water 330ml'

  CANTIDAD:
    -> Predicha: 126.14 unidades


